# Distillation for LLMs, off-policy, on-policy and logits

LLM Training Lab, module 20. Regime DISTILLATION: teacher Qwen3-4B-Base in 4-bit NF4 (frozen), student Qwen3-0.6B-Base with LoRA adapters, trained on GSM8K questions three ways.

Run the cells in order on one T4 (Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2). Steps 1 and 2 also run on a CPU. Code cell numbers match the lab page.

In [ ]:
import os, sys, platform, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # one GPU; set before torch loads
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
problems = []
if sys.version_info < (3, 11):
    problems.append("Python 3.11 or newer is needed (3.12 recommended). Start a current Colab or Kaggle session.")
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:                                   # a CPU runtime has no nvidia-smi
    smi = []
gpu = smi[0] if smi else ""
if not gpu:
    problems.append("No GPU: steps 1 and 2 still run. For step 3 on, Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2.")
elif "P100" in gpu:
    problems.append("A P100 has no kernels in current PyTorch. Pick a T4.")
cache = os.path.expanduser("~/.cache")
os.makedirs(cache, exist_ok=True)
free_disk = shutil.disk_usage(cache).free / 1e9
if free_disk < 15:
    problems.append("Free at least 15 GB of disk for the two models and GSM8K (" + str(round(free_disk)) + " GB free).")
try:
    torch_version = version("torch")
except PackageNotFoundError:
    torch_version = "not installed"
print("Python", platform.python_version(), "| GPU", gpu or "none", "| torch", torch_version, "| disk free", round(free_disk), "GB")
for p in problems:
    print("PROBLEM", p)
print("Runtime OK" if not problems else "Read the PROBLEM lines first.")

In [ ]:
# Pinned Hugging Face stack of our recorded run. torch and numpy stay as the runtime ships them.
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 trl==1.17.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 bitsandbytes==0.50.2 tokenizers==0.23.3 huggingface_hub==2.4.0

In [ ]:
import sys, time
class _Tee:
    def __init__(self, path):
        self.f, self.o = open(path, "a"), sys.stdout
    def write(self, x):
        self.o.write(x); self.f.write(x); self.f.flush()
    def flush(self):
        self.o.flush(); self.f.flush()
    def __getattr__(self, name):          # isatty, fileno, encoding... come from the real stream
        return getattr(self.o, name)
sys.stdout = _Tee("notebook_stdout.txt")
T_START = time.time()

## Step 1. Fit one Gaussian to two modes, both ways (NumPy, CPU)

Code cell 1 on the lab page.

In [ ]:
print("CELL cell01_step1 start", round(time.time() - T_START, 1))
import numpy as np

x = np.linspace(-6, 6, 1201)
dx = x[1] - x[0]

def normal(mu, sd):
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))

def kl(a, b):
    # KL(a || b) on the grid, in nats. The tiny floor stops log(0) where a density underflows.
    return float(np.sum(a * (np.log(a + 1e-300) - np.log(b + 1e-300))) * dx)

teacher = 0.6 * normal(-2.0, 0.6) + 0.4 * normal(2.0, 0.6)       # two modes, 60% and 40% of the mass
grid = [(mu, sd) for mu in np.arange(-4, 4.001, 0.05) for sd in np.arange(0.2, 4.001, 0.05)]
fits = {"forward KL(teacher || student)": min(grid, key=lambda g: kl(teacher, normal(*g))),
        "reverse KL(student || teacher)": min(grid, key=lambda g: kl(normal(*g), teacher))}

valley = np.abs(x) < 0.5                                          # the gap between the modes, where the teacher has little mass
print("teacher mass in the valley |x| < 0.5:", round(float(teacher[valley].sum() * dx), 4))
for name, (mu, sd) in fits.items():
    s = normal(mu, sd)
    print(name, "| best Gaussian: mean", round(mu, 2), "sd", round(sd, 2),
          "| student mass in the valley", round(float(s[valley].sum() * dx), 4),
          "| forward KL", round(kl(teacher, s), 3), "| reverse KL", round(kl(s, teacher), 3))
print("reverse KL with the student on the other mode, mean 2.0 sd 0.6:", round(kl(normal(2.0, 0.6), teacher), 3))
print("CELL cell01_step1 end", round(time.time() - T_START, 1))

**What you should see.** The forward-KL fit is mean -0.4 and standard deviation 2.05, with 0.1873 of its mass in the valley |x| < 0.5, against 0.006 for the teacher. The reverse-KL fit is mean -2.0 and standard deviation 0.6, with 0.006 in the valley and a reverse KL of 0.51. On the other mode, mean 2.0, the reverse KL is 0.915.

## Step 2. Write the divergence and match TRL's kernel (CPU)

Code cell 2 on the lab page.

In [ ]:
print("CELL cell02_step2 start", round(time.time() - T_START, 1))
import math
import torch
from trl.kernels.chunked_divergence import ChunkedDivergenceFunction   # the kernel inside TRL's DistillationTrainer

def divergence(student_logits, teacher_logits, beta):
    # Per-token divergence over the whole vocabulary. beta 0: forward KL(teacher || student).
    # beta 1: reverse KL(student || teacher). In between: generalized JSD with mixture m = beta * teacher + (1 - beta) * student.
    lps, lpt = student_logits.float().log_softmax(-1), teacher_logits.float().log_softmax(-1)
    if beta == 0:
        return (lpt.exp() * (lpt - lps)).sum(-1)
    if beta == 1:
        return (lps.exp() * (lps - lpt)).sum(-1)
    m = torch.logsumexp(torch.stack([lps + math.log(1 - beta), lpt + math.log(beta)]), 0)
    return beta * (lpt.exp() * (lpt - m)).sum(-1) + (1 - beta) * (lps.exp() * (lps - m)).sum(-1)

# TRL never builds the [tokens, vocab] logits. It takes hidden states and each model's output head and works in tiles.
torch.manual_seed(0)
V, N = 151936, 6                                  # Qwen3's vocabulary size, 6 token positions
h_s, h_t = torch.randn(N, 64), torch.randn(N, 96) # student and teacher hidden sizes differ; only the vocabulary is shared
W_s, W_t = torch.randn(V, 64) * 0.1, torch.randn(V, 96) * 0.1
for beta in (0.0, 0.5, 1.0):
    a, b = h_s.clone().requires_grad_(True), h_s.clone().requires_grad_(True)
    ours = divergence(a @ W_s.T, h_t @ W_t.T, beta)
    ours.sum().backward()
    theirs, _ = ChunkedDivergenceFunction.apply(b, W_s, None, h_t, W_t, None, beta)
    theirs.sum().backward()
    print("beta", beta, "| ours", round(ours.mean().item(), 6), "| TRL", round(theirs.mean().item(), 6),
          "| largest gap", "%.1e" % float((ours - theirs).detach().abs().max()),
          "| largest gradient gap", "%.1e" % float((a.grad - b.grad).abs().max()))
print("CELL cell02_step2 end", round(time.time() - T_START, 1))

**What you should see.** For beta 0, 0.5 and 1 your values and TRL's agree to five decimals, 0.851494, 0.176187 and 0.853843 on our laptop, where TRL printed 0.853842 for beta 1. The largest value gap is 3.1e-06 or less, and the largest gradient gap 1.2e-07 or less. On the T4 the gaps were 2.4e-06 and 4.1e-08.

## Step 3. Load the teacher and the student, and measure both on GSM8K

Code cell 3 on the lab page.

In [ ]:
print("CELL cell03_step3 start", round(time.time() - T_START, 1))
import gc, json, random, re, time
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig

GPU = torch.device("cuda")
TEACHER, STUDENT = "Qwen/Qwen3-4B-Base", "Qwen/Qwen3-0.6B-Base"
N_TEST, N_TRAIN = 500, 2048
NL = chr(10)
STOPS = [NL + "Question", NL + "[Question]"]   # a base model answers, then starts a new question; stop there

tok = AutoTokenizer.from_pretrained(STUDENT)
tok.padding_side = "left"          # generation pads prompts on the left
same_vocab = AutoTokenizer.from_pretrained(TEACHER).get_vocab() == tok.get_vocab()
nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)
teacher = AutoModelForCausalLM.from_pretrained(TEACHER, quantization_config=nf4, dtype=torch.float16, device_map={"": 0}).eval()
print("teacher on GPU", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
student = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.float16).to(GPU).eval()
print("vocab size, teacher", teacher.config.vocab_size, "| student", student.config.vocab_size,
      "| tokenizer entries", len(tok), "| same token ids", same_vocab)

gsm = load_dataset("openai/gsm8k", "main")
test = [dict(q=e["question"], a=e["answer"]) for e in gsm["test"].select(range(N_TEST))]
train = [dict(q=e["question"], a=e["answer"]) for e in gsm["train"].shuffle(seed=0).select(range(N_TRAIN))]

def prompt(q):
    return "Question: " + q + NL + "Answer:"

def number(s):
    s = s.replace(",", "").rstrip(".")
    try:
        return float(s)
    except ValueError:
        return None

def cut(text):
    # the reply up to the first stop string
    ends = [text.find(s) for s in STOPS if s in text]
    return text[:min(ends)] if ends else text

def extract(text):
    # the number after the last "answer is" or "####", else the last number in the reply
    text = cut(text)
    found = re.findall(r"(?:answer is|####)[^0-9-]*(-?[0-9][0-9,]*(?:[.][0-9]+)?)", text)
    found = found or re.findall(r"-?[0-9][0-9,]*(?:[.][0-9]+)?", text)
    return number(found[-1]) if found else None

def gold(answer):
    return number(answer.split("####")[-1].strip())

def generate(model, prompts, sample=False, batch=64, max_new=256):
    # returns, per prompt, the generated token ids up to and including the stop string (pads removed)
    cfg = GenerationConfig(max_new_tokens=max_new, do_sample=sample, temperature=1.0 if sample else None, top_k=0 if sample else None,
                           top_p=1.0 if sample else None, stop_strings=STOPS, pad_token_id=tok.pad_token_id,
                           eos_token_id=tok.eos_token_id)
    out = []
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True).to(GPU)
        with torch.no_grad():
            g = model.generate(**enc, generation_config=cfg, tokenizer=tok)
        for row in g[:, enc["input_ids"].shape[1]:].tolist():
            out.append([t for t in row if t != tok.pad_token_id])
    return out

test_prompt_ids = [tok(prompt(x["q"]))["input_ids"] for x in test]
train_prompt_ids = [tok(prompt(x["q"]))["input_ids"] for x in train]
print("test", len(test), "| train", len(train), "| longest prompt", max(len(p) for p in test_prompt_ids + train_prompt_ids), "tokens")
print("CELL cell03_step3 end", round(time.time() - T_START, 1))

In [ ]:
print("CELL cell04_step3 start", round(time.time() - T_START, 1))
def pack(prompt_ids, comp_ids):
    # right-padded rows of prompt + completion, an attention mask, and a mask over the completion tokens
    L = max(len(p) + len(c) for p, c in zip(prompt_ids, comp_ids))
    ids = torch.full((len(prompt_ids), L), tok.pad_token_id)
    att, cm = torch.zeros_like(ids), torch.zeros_like(ids)
    for i, (p, c) in enumerate(zip(prompt_ids, comp_ids)):
        ids[i, :len(p) + len(c)] = torch.tensor(p + c)
        att[i, :len(p) + len(c)] = 1
        cm[i, len(p):len(p) + len(c)] = 1
    return ids.to(GPU), att.to(GPU), cm.to(GPU)

def logits_at(model, ids, att, cm):
    # fp32 logits only at the positions that predict a completion token: [tokens, vocab], never [rows, length, vocab]
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    with torch.autocast("cuda", dtype=torch.float16):
        h = base.model(input_ids=ids, attention_mask=att).last_hidden_state
        return base.lm_head(h[:, :-1][cm[:, 1:].bool()]).float()

def kl_report(model, prompt_ids, comp_ids, micro=8):
    # mean per-token forward KL, reverse KL and student entropy over the completion tokens, in nats
    tot = dict(forwardKL=0.0, reverseKL=0.0, studentEntropy=0.0)
    n = 0
    for i in range(0, len(prompt_ids), micro):
        ids, att, cm = pack(prompt_ids[i:i + micro], comp_ids[i:i + micro])
        with torch.no_grad():
            lps = logits_at(model, ids, att, cm).log_softmax(-1)
            lpt = logits_at(teacher, ids, att, cm).log_softmax(-1)
            tot["forwardKL"] += float((lpt.exp() * (lpt - lps)).sum())
            tot["reverseKL"] += float((lps.exp() * (lps - lpt)).sum())
            tot["studentEntropy"] += float(-(lps.exp() * lps).sum())
        n += int(cm[:, 1:].sum())
        del lps, lpt
    return {k: round(v / n, 4) for k, v in tot.items()}

R = {}

def evaluate(model, name, batch=100):
    # greedy answers to the test questions, accuracy, and KL to the teacher on the model's own answers and on the teacher's
    t0 = time.time()
    comp = generate(model, [prompt(x["q"]) for x in test], batch=batch)
    gen_sec = time.time() - t0
    texts = [tok.decode(c, skip_special_tokens=True) for c in comp]
    correct = [extract(t) is not None and extract(t) == gold(x["a"]) for t, x in zip(texts, test)]
    rec = dict(acc=round(float(np.mean(correct)), 4), correct=correct, stopped=round(float(np.mean([cut(t) != t for t in texts])), 3),
               meanTokens=round(float(np.mean([len(c) for c in comp])), 1), genSec=round(gen_sec, 1), texts=texts)
    if model is not teacher:
        rec["onOwnAnswers"] = kl_report(model, test_prompt_ids, comp)
        rec["onTeacherAnswers"] = kl_report(model, test_prompt_ids, teacher_test)
    R[name] = rec
    json.dump(R, open("distill_results.json", "w"))
    print(name, "| accuracy", rec["acc"], "| stopped", rec["stopped"], "| mean answer tokens", rec["meanTokens"],
          "| generation s", rec["genSec"], "| own answers", rec.get("onOwnAnswers"), "| teacher answers", rec.get("onTeacherAnswers"))
    return comp

teacher_test = evaluate(teacher, "teacher", batch=64)
print(R["teacher"]["texts"][0][:600])
evaluate(student, "student, before training")
print(R["student, before training"]["texts"][0][:600])
print("CELL cell04_step3 end", round(time.time() - T_START, 1))

**What you should see.** The teacher takes 2.65 GB. Both vocabulary sizes are 151,936, the tokenizer has 151,669 entries and the token ids are the same. The longest prompt is 198 tokens. The teacher scores 0.756 and the student 0.522, with the KL numbers in the table.

## Step 4. Let the teacher write the training answers

Code cell 5 on the lab page.

In [ ]:
print("CELL cell05_step4 start", round(time.time() - T_START, 1))
USE_OUR_TEACHER_DATA = True    # False: the teacher writes all 2,048 training answers on your T4 instead of downloading ours

if USE_OUR_TEACHER_DATA:
    from huggingface_hub import hf_hub_download
    ours = json.load(open(hf_hub_download("aimlcompanion/llm-training-lab", "lltDistillation/teacher_train.json")))
    assert ours["questions"] == [x["q"] for x in train], "the shuffle differs from ours; keep seed=0 and N_TRAIN = 2048"
    teacher_train = ours["completionIds"]
else:
    t0 = time.time()
    teacher_train = generate(teacher, [prompt(x["q"]) for x in train], batch=64)
    sec = time.time() - t0
    new_tokens = sum(len(c) for c in teacher_train)
    print("teacher wrote", new_tokens, "tokens in", round(sec), "s |", round(new_tokens / sec), "tokens/s")
    json.dump({"questions": [x["q"] for x in train], "completionIds": teacher_train}, open("teacher_train.json", "w"))

texts = [tok.decode(c, skip_special_tokens=True) for c in teacher_train]
right = [extract(t) is not None and extract(t) == gold(x["a"]) for t, x in zip(texts, train)]
print("teacher answers", len(teacher_train), "| correct", round(float(np.mean(right)), 3), "| ended at the stop string",
      round(float(np.mean([cut(t) != t for t in texts])), 3), "| mean tokens", round(float(np.mean([len(c) for c in teacher_train])), 1))

# the human-written GSM8K solutions, given the same ending so every training target stops the same way
ref_train = [tok(" " + x["a"] + STOPS[0] + ":")["input_ids"] for x in train]
print("reference solutions, mean tokens", round(float(np.mean([len(c) for c in ref_train])), 1))
print(texts[0])
print("CELL cell05_step4 end", round(time.time() - T_START, 1))

**What you should see.** The cell prints 2,048 answers, 0.817 correct, 0.893 ending at the stop string and 96.4 tokens on average. The reference solutions average 125.7 tokens. With `USE_OUR_TEACHER_DATA = False` it also prints the token count, the seconds and the tokens per second.

## Step 5. Off-policy, SFT on the teacher's answers

Code cell 6 on the lab page.

In [ ]:
print("CELL cell06_step5 start", round(time.time() - T_START, 1))
import math, os
from peft import LoraConfig, get_peft_model

STEPS, BATCH, MICRO, LR = 64, 32, 4, 2e-4      # 64 steps of 32 questions: each arm sees the same 2,048 training questions

def new_student():
    # frozen fp16 Qwen3-0.6B-Base with fp32 LoRA adapters, rank 16, on every linear layer
    m = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.float16).to(GPU)
    for p in m.parameters():
        p.requires_grad = False
    m.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    m.enable_input_require_grads()
    m = get_peft_model(m, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", task_type="CAUSAL_LM"))
    for p in m.parameters():
        if p.requires_grad:
            p.data = p.data.float()
    return m

def sync(model, copy):
    # write base + LoRA into the plain fp16 student, which then samples and evaluates without adapter layers
    with torch.no_grad():
        target = dict(copy.named_modules())
        for name, mod in model.get_base_model().named_modules():
            if hasattr(mod, "lora_A") and "default" in mod.lora_A:
                delta = (mod.lora_B["default"].weight.float() @ mod.lora_A["default"].weight.float()) * mod.scaling["default"]
                target[name].weight.copy_((mod.base_layer.weight.float() + delta).half())

def token_loss(kind, s_logits, t_logits, targets):
    lps = s_logits.log_softmax(-1)
    if kind == "sft":                                   # cross-entropy on the token that was written
        return -lps.gather(1, targets[:, None]).squeeze(1)
    lpt = t_logits.log_softmax(-1)
    if kind == "forward":                               # KL(teacher || student) over the whole vocabulary
        return (lpt.exp() * (lpt - lps)).sum(-1)
    return (lps.exp() * (lps - lpt)).sum(-1)            # "reverse": KL(student || teacher)

def train_student(kind, source, seed=0, steps=STEPS):
    # source "reference" or "teacher": fixed answers (off-policy). "student": answers sampled from the student every step (on-policy)
    torch.manual_seed(seed)
    model = new_student()
    trainable = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.0)
    warm = 3
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda", init_scale=2048)
    order = list(range(len(train)))
    random.Random(seed).shuffle(order)
    torch.cuda.reset_peak_memory_stats()
    losses, t0, sample_sec = [], time.time(), 0.0
    for step in range(steps):
        idx = order[step * BATCH:(step + 1) * BATCH]
        P = [train_prompt_ids[i] for i in idx]
        if source == "student":
            sync(model, student)
            g0 = time.time()
            C = generate(student, [prompt(train[i]["q"]) for i in idx], sample=True, batch=BATCH)
            sample_sec += time.time() - g0
        else:
            C = [(teacher_train if source == "teacher" else ref_train)[i] for i in idx]
        n_tok = sum(len(c) for c in C)
        model.train()
        opt.zero_grad(set_to_none=True)
        total = 0.0
        for k in range(0, len(P), MICRO):
            ids, att, cm = pack(P[k:k + MICRO], C[k:k + MICRO])
            s = logits_at(model, ids, att, cm)
            t = None
            if kind != "sft":
                with torch.no_grad():
                    t = logits_at(teacher, ids, att, cm)
            loss = token_loss(kind, s, t, ids[:, 1:][cm[:, 1:].bool()]).sum() / n_tok
            scaler.scale(loss).backward()
            total += loss.item()
            del s, t, loss
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        losses.append(round(total, 4))
        if step % 16 == 0 or step == steps - 1:
            print("step", step + 1, "of", steps, "| loss", round(total, 4), "| answer tokens", n_tok, "| seconds", round(time.time() - t0))
    stats = dict(trainSec=round(time.time() - t0, 1), sampleSec=round(sample_sec, 1),
                 peakGB=round(torch.cuda.max_memory_allocated() / 1e9, 2), loss=losses)
    return model, stats

def run_arm(name, kind, source, seed=0):
    # train one arm, save its adapter, evaluate it on the test questions, then free the GPU
    model, stats = train_student(kind, source, seed)
    model.save_pretrained(os.path.join("adapters", re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")))
    sync(model, student)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    evaluate(student, name)
    R[name].update(stats)
    json.dump(R, open("distill_results.json", "w"))

run_arm("SFT on teacher answers", "sft", "teacher")
print("CELL cell06_step5 end", round(time.time() - T_START, 1))

**What you should see.** The loss goes 0.3922 at step 1, 0.2798 at step 17 and 0.2914 at step 64, in 248 s, peaking at 7.03 GB. The student scores 0.48, stops 0.932 of its replies and writes 99.2 tokens on average. Its reverse KL on its own answers is 0.1739 and its entropy 0.2784.

## Step 6. Logit KD, forward KL over the whole vocabulary

Code cell 7 on the lab page.

In [ ]:
print("CELL cell07_step6 start", round(time.time() - T_START, 1))
run_arm("Logit KD, forward KL on teacher answers", "forward", "teacher")
for name in ["SFT on teacher answers", "Logit KD, forward KL on teacher answers"]:
    r = R[name]
    print(name, "| accuracy", r["acc"], "| loss at step 1", r["loss"][0], "| at step", len(r["loss"]), r["loss"][-1],
          "| training s", r["trainSec"], "| peak GB", r["peakGB"])
print("CELL cell07_step6 end", round(time.time() - T_START, 1))

**What you should see.** The loss goes 0.2631 at step 1, 0.1723 at step 17 and 0.1893 at step 64, in 573 s at 8.32 GB. The student scores 0.518, with a forward KL on the teacher's answers of 0.1572 and an entropy of 0.4966. The comparison lines print 0.48 for SFT and 0.518 for logit KD.

## Step 7. On-policy, the student samples and the teacher grades every token

Code cell 8 on the lab page.

In [ ]:
print("CELL cell08_step7 start", round(time.time() - T_START, 1))
USE_OUR_ADAPTER = True    # True loads our trained adapter from the Hub instead of training this arm on your T4
name = "On-policy, reverse KL on student samples"

if USE_OUR_ADAPTER:
    from huggingface_hub import snapshot_download
    from peft import PeftModel
    folder = snapshot_download("aimlcompanion/llm-training-lab", allow_patterns=["lltDistillation/onpolicy-reverse-s0/*"])
    model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(STUDENT, dtype=torch.float16).to(GPU),
                                      os.path.join(folder, "lltDistillation", "onpolicy-reverse-s0"))
    sync(model, student)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    evaluate(student, name)
else:
    run_arm(name, "reverse", "student")
    r = R[name]
    print("sampling", r["sampleSec"], "s of", r["trainSec"], "s training | loss at step 1", r["loss"][0], "| at step", len(r["loss"]), r["loss"][-1])
print("CELL cell08_step7 end", round(time.time() - T_START, 1))

In [ ]:
print("CELL cell09_step7 start", round(time.time() - T_START, 1))
from datasets import Dataset
from trl import DistillationConfig, DistillationTrainer

# The same objective in TRL's stable trainer: it samples from the student and applies the divergence of step 2 (beta 1 = reverse KL).
ds = Dataset.from_list([{"prompt": prompt(x["q"])} for x in train[:24]])
cfg = DistillationConfig(output_dir="trl-distill", max_steps=3, per_device_train_batch_size=8, learning_rate=LR, beta=1.0,
                         temperature=1.0, max_completion_length=256, fp16=True, bf16=False, gradient_checkpointing=True,
                         logging_steps=1, save_strategy="no", report_to="none",
                         model_init_kwargs={"dtype": "float32"})                   # dtype as a string, not torch.float32
trainer = DistillationTrainer(model=STUDENT, teacher_model=teacher, args=cfg, train_dataset=ds, processing_class=tok,
                              peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear",
                                                     task_type="CAUSAL_LM"))
print("trainable dtypes", {str(p.dtype) for p in trainer.model.parameters() if p.requires_grad})
t0 = time.time()
trainer.train()
for h in trainer.state.log_history[:3]:
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in h.items() if k in ("step", "loss", "entropy", "grad_norm")})
print("3 steps in", round(time.time() - t0, 1), "s")
del trainer
gc.collect()
torch.cuda.empty_cache()
print("CELL cell09_step7 end", round(time.time() - T_START, 1))

**What you should see.** The loss goes 0.357 at step 1, 0.2727 at step 17 and 0.2409 at step 64, after 1,496 s of which 894.5 s were sampling. The student scores 0.532, and its reverse KL on its own answers is 0.1503, the lowest of any method. TRL prints float32 trainable weights and three losses, 0.5307, 0.46 and 0.423, in 76.7 s.

## Step 8. Separate the source of the answers from the KL direction (RUN_EXPERIMENT)

Code cell 10 on the lab page.

In [ ]:
print("CELL cell10_step8 start", round(time.time() - T_START, 1))
RUN_EXPERIMENT = False    # True trains the other three arms of the grid and a second seed of the first three on your T4

grid = [("SFT on reference answers", "sft", "reference", 0),
        ("Logit KD, reverse KL on teacher answers", "reverse", "teacher", 0),
        ("On-policy, forward KL on student samples", "forward", "student", 0),
        ("SFT on teacher answers, seed 1", "sft", "teacher", 1),
        ("Logit KD, forward KL on teacher answers, seed 1", "forward", "teacher", 1),
        ("On-policy, reverse KL on student samples, seed 1", "reverse", "student", 1)]
if RUN_EXPERIMENT:
    for name, kind, source, seed in grid:
        run_arm(name, kind, source, seed)

print("arm | accuracy | reverse KL on own answers | forward KL on teacher answers | student entropy on own answers")
for name, r in R.items():
    if name != "teacher":
        print(name, "|", r["acc"], "|", r["onOwnAnswers"]["reverseKL"], "|", r["onTeacherAnswers"]["forwardKL"], "|",
              r["onOwnAnswers"]["studentEntropy"])
print("CELL cell10_step8 end", round(time.time() - T_START, 1))

**What you should see.** With `RUN_EXPERIMENT = True` the cell trains six more students, 123 min on a T4 in our run, and prints the table above. With the default `False` it prints the table for the students you trained in steps 5 to 7.

## Step 9. Compare the methods with paired intervals and two seeds

Code cell 11 on the lab page.

In [ ]:
print("CELL cell11_step9 start", round(time.time() - T_START, 1))
def paired(a, b, reps=10000):
    # accuracy of arm a minus arm b on the same test questions, with a 95% paired bootstrap interval
    x, y = np.array(R[a]["correct"], float), np.array(R[b]["correct"], float)
    idx = np.random.default_rng(0).integers(0, len(x), (reps, len(x)))
    d = (x[idx] - y[idx]).mean(1)
    return round(float((x - y).mean()), 3), round(float(np.percentile(d, 2.5)), 3), round(float(np.percentile(d, 97.5)), 3)

base, seqkd = "student, before training", "SFT on teacher answers"
print("teacher accuracy", R["teacher"]["acc"], "| student before training", R[base]["acc"])
for name, r in R.items():
    if name in ("teacher", base):
        continue
    line = [name, "| accuracy", r["acc"], "| vs before", paired(name, base)]
    if name != seqkd:
        line += ["| vs SFT on teacher answers", paired(name, seqkd)]
    if "trainSec" in r:
        line += ["| training s", r["trainSec"], "of it sampling", r["sampleSec"]]
    print(*line)
print("CELL cell11_step9 end", round(time.time() - T_START, 1))

**What you should see.** The cell prints the teacher at 0.756 and the untrained student at 0.522. Then each method gets one line, with its accuracy, its difference from the untrained student and from SFT on teacher answers, and its training seconds. The intervals match the table.

In [ ]:
import torch, json
print("KERNEL peak GB over the whole session", round(torch.cuda.max_memory_allocated() / 1e9, 2))